In [1]:
import sys # for automation and parallelization: set manual to false when run by a launcher
import json
MANUAL, SCENARIO = (True, 'base') if 'ipykernel' in sys.argv[0]\
     else (False,  json.loads(sys.argv[1])['scenario'])

In [2]:
import sys
sys.path.insert(0, r'../../../quetzal')

from quetzal.model import stepmodel, connectionscanmodel
from quetzal.io import excel
from quetzal.io import road as road_io
import os
import numpy as np
import pandas as pd
import geopandas as gpd
if MANUAL:
    %matplotlib inline

io_engine= 'pyogrio'

In [3]:
qpath = r'../../'
ipath = qpath + r'inputs/'
mpath = qpath + r'scenarios/{}/models/'.format(SCENARIO)

In [4]:
var = excel.read_var(ipath + r'parameters.xlsx', scenario=SCENARIO)

# Create model

In [5]:
UTM_EPSG = 2154  # France
if var['run_mode']['mode'] == 'frequency':
    model = stepmodel.StepModel(epsg=UTM_EPSG, coordinates_unit='meter')
elif var['run_mode']['mode'] == 'timetables':
    model = connectionscanmodel.ConnectionScanModel(epsg=UTM_EPSG, coordinates_unit='meter')

In [6]:
PERCEIVED_HEADWAY = var['perceived_headway']['perceived_headway']

BIKE_SPEED = var['bike_speed'].to_dict()
BIKE_FACTORS = {int(x.replace('k', '')): (1+y) for x, y in var['bike_factor'].to_dict().items()}

In [7]:
def read_file(path,engine):
    gdf = gpd.read_file(path, engine=engine)
    if 'index' in gdf.columns:
        gdf = gdf.set_index('index')
    else:
        gdf.index.name = 'index'
    return gdf

def read_links(filename, engine):
    links = read_file(filename, engine=engine)
    with open(filename, 'r') as j:
        json_data = json.loads(j.read())
    json_prop = json_data['features'][0]['properties'].keys()
    missing_columns = set(json_prop).difference(links.columns)
    for col in missing_columns:
        if col != 'index':
            d = {f['properties']['index']: f['properties'].get(col) for f in json_data['features']}
            links[col] = links.index.map(d) 

    return links

In [8]:
model.links = read_links(ipath + f'pt/links.geojson',  engine=io_engine).to_crs(epsg=2154)
model.nodes = read_file(ipath + f'pt/nodes.geojson', engine=io_engine).to_crs(epsg=2154)

model.road_nodes = read_file(ipath + 'road/simple_rn.geojson', engine=io_engine).to_crs(epsg=2154)
model.road_links = read_file(ipath + 'road/simple_rl.geojson', engine=io_engine).to_crs(epsg=2154)

# Edit PT network

In [9]:
model.links.drop_duplicates(subset=['a', 'b', 'trip_id', 'link_sequence'], keep='first', inplace=True)

In [10]:
pd.set_option('display.max_columns', None)

### Fix times and speeds, ignore irrelavant modes

In [ ]:
model.links['speed'] = model.links['length'] / model.links['time']
model.links = model.links[~model.links['route_type'].isin(['schoolbus', 'ferry'])]
model.nodes = model.nodes.loc[model.nodes.index.isin(model.links['a'].unique().tolist() + model.links['b'].unique().tolist())]
model.integrity_fix_sequences()

dropped broken sequences: ['78_TBM_3_0']


### Set perceived headways

In [ ]:
equivaliences={
        'tramway': {0:0, 1800: 1800, 3600: 2400,5400: 2700, 7200: 3000, 14400: 3300},
        'bus': {0:0, 1800: 1800, 3600: 2400,5400: 2700, 7200: 3000, 14400: 3300},
        'rail': {0:0, 1800: 1800, 3600: 2100,5400: 2400, 7200: 2700, 14400: 3000}
    }

from bisect import bisect_right

def perceived_headway_mode(mode):
    e=equivaliences[mode]
    if not e:
        raise ValueError("Le dictionnaire ne doit pas être vide.")

    points = sorted(e.items())
    xs = [x for x, _ in points]
    ys = [y for _, y in points]

    def f(x):
        if len(xs) == 1:
            return ys[0]

        # Choisit le segment correspondant, ou le premier/dernier pour extrapoler.
        i = max(0, min(bisect_right(xs, x) - 1, len(xs) - 2))

        x0, y0 = xs[i], ys[i]
        x1, y1 = xs[i + 1], ys[i + 1]

        return y0 + (x - x0) * (y1 - y0) / (x1 - x0)

    return f

In [ ]:
model.links['perceived_headway'] = model.links['headway'].copy()
model.links.loc[model.links['route_type']=='rail', 'perceived_headway'] = model.links['headway'].apply(perceived_headway_mode('rail'))
model.links.loc[model.links['route_type']=='bus', 'perceived_headway'] = model.links['headway'].apply(perceived_headway_mode('bus'))
model.links.loc[model.links['route_type']=='tramway', 'perceived_headway'] = model.links['headway'].apply(perceived_headway_mode('tramway'))

# Edit road network

In [ ]:
### TODO: Adapt bike_speed using Royston's input

model.road_links['speed'] = model.road_links['speed'].fillna(8)
model.road_links['time'] = model.road_links['length']/model.road_links['speed']*3.6

In [ ]:
model.road_links['capacity'] = model.road_links['lanes'].fillna(1)*2000

# EXPORT

In [13]:
model.to_zippedpickles(mpath + r'all_networks')

broken_sequences: 100%|██████████| 11/11 [00:01<00:00,  8.16it/s]


In [14]:
end_of_notebook

NameError: name 'end_of_notebook' is not defined

In [ ]:
sm.integrity_test_all(errors='ignore')

passed: integrity_test_circular_lines
not performed: integrity_test_collision


8106it [00:01, 5331.47it/s]


passed: integrity_test_dead_ends
not performed: integrity_test_duplicate_volumes
passed: integrity_test_isolated_roads
passed: integrity_test_nodeset_consistency


8106it [00:03, 2505.38it/s]


passed: integrity_test_road_network
passed: integrity_test_road_nodeset_consistency
passed: integrity_test_sequences


In [ ]:
brt_links = sm.links.loc[sm.links['route_type']=='brt']
brt_links.groupby('route_id')['length'].sum() / brt_links.groupby('route_id')['time'].sum() * 3.6

Series([], dtype: float64)